In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from pathlib import Path

from torchvision import transforms
from torch.utils.data import DataLoader

import torch
from tqdm import tqdm

from src.datasets.celeba_spoof_dataset import (
    CelebASpoofDataset,
    crop_face
)

from src.models.fsfm_lite import FSFMLite

In [ ]:
test_df = pd.read_csv(
    METADATA_ROOT / "test_df.csv"
)

test_df["image_path"] = test_df["image_rel_path"].apply(
    lambda x: str(DATA_ROOT / x)
)

test_df["bb_path"] = test_df["image_rel_path"].apply(
    lambda x: str(
        DATA_ROOT /
        x.replace(".jpg","_BB.txt")
         .replace(".png","_BB.txt")
    )
)

In [ ]:
model = FSFMLite()

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

model.eval()

In [ ]:
face_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_dataset = CelebASpoofDataset(
    test_df,
    transform=face_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [ ]:
results = []

idx_offset = 0

with torch.no_grad():

    for batch in tqdm(test_loader):

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model(images)

        probs = torch.softmax(
            logits,
            dim=1
        )[:,1]

        preds = logits.argmax(
            dim=1
        )

        batch_size = labels.size(0)

        for i in range(batch_size):

            results.append({

                "index":
                    idx_offset + i,

                "label":
                    int(labels[i]),

                "pred":
                    int(preds[i]),

                "prob":
                    float(probs[i])

            })

        idx_offset += batch_size

In [ ]:
result_df = pd.DataFrame(
    results
)

result_df.head()

In [ ]:
false_negative = result_df[
    (result_df.label == 1)
    &
    (result_df.pred == 0)
]

print(
    len(false_negative)
)

In [ ]:
false_negative = false_negative.merge(

    test_df,

    left_on="index",
    right_index=True

)

false_negative.head()

In [ ]:
fig, axes = plt.subplots(
    5,
    5,
    figsize=(15,15)
)

for ax, (_, row) in zip(
    axes.flatten(),
    false_negative.head(25).iterrows()
):

    face = crop_face(
        row.image_path,
        row.bb_path
    )

    ax.imshow(face)

    ax.set_title(
        f"P={row.prob:.2f}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
hard_cases = false_negative.sort_values(
    "prob",
    ascending=True
)

hard_cases.head()

In [ ]:
fig, axes = plt.subplots(
    5,
    5,
    figsize=(15,15)
)

for ax, (_, row) in zip(
    axes.flatten(),
    hard_cases.head(25).iterrows()
):

    face = crop_face(
        row.image_path,
        row.bb_path
    )

    ax.imshow(face)

    ax.set_title(
        f"{row.prob:.4f}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
false_negative.to_csv(

    REPORT_ROOT
    / "false_negative.csv",

    index=False
)

print(
    false_negative.shape
)